# Embedding Similarity

Local mode asks whether paraphrases are closer to their edit than same-property locality prompts inside one record. Global mode is the decision experiment for an external-memory router and is intentionally shown as a sequence of questions: can the correct edit be retrieved, do positive and no-edit scores separate, why the previous positive-only threshold produced false activations, whether explicit locality limits repair it, and whether top1/top2 margin adds useful evidence. Tables state the question before plots show scaling.

## Environment

In [ ]:
from contextlib import redirect_stderr
import gc
import hashlib
from io import StringIO
import json
import sys
from pathlib import Path
import unicodedata
from typing import Any

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, load_from_disk
from IPython.display import display
from tqdm.auto import tqdm


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_REPO = "Savassan/KnowledgeSurgeon"
DATASET_CONFIG = "similarity"
DATASET_SPLIT = "data"
LOCAL_DATASET_PATH = PROJECT_ROOT / "data" / "datasets" / "uniedit" / "final"
OUTPUT_DIR = PROJECT_ROOT / "data" / "similarity" / "embedding"
REPRESENTATION_TYPE = "normalized_dense_sentence_embedding"

EMBEDDING_MODEL_SPECS = {
    "all-mpnet-base-v2": {
        "repo": "sentence-transformers/all-mpnet-base-v2",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "all-MiniLM-L6-v2": {
        "repo": "sentence-transformers/all-MiniLM-L6-v2",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "all-MiniLM-L12-v2": {
        "repo": "sentence-transformers/all-MiniLM-L12-v2",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "e5-base-v2": {
        "repo": "intfloat/e5-base-v2",
        "backend": "sentence_transformers",
        "prefixes": {
            "symmetric": "query: ",
            "query": "query: ",
            "passage": "passage: ",
        },
    },
    "e5-large-v2": {
        "repo": "intfloat/e5-large-v2",
        "backend": "sentence_transformers",
        "prefixes": {
            "symmetric": "query: ",
            "query": "query: ",
            "passage": "passage: ",
        },
    },
    "bge-base-en-v1.5": {
        "repo": "BAAI/bge-base-en-v1.5",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "bge-large-en-v1.5": {
        "repo": "BAAI/bge-large-en-v1.5",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "bge-m3": {
        "repo": "BAAI/bge-m3",
        "backend": "flag_embedding",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
}


def resolve_embedding_model_keys(model_keys: list[str]) -> list[str]:
    selected = list(dict.fromkeys(model_keys))
    if not selected:
        raise ValueError("embedding_model_keys must contain at least one model.")
    unknown = [key for key in selected if key not in EMBEDDING_MODEL_SPECS]
    if unknown:
        raise ValueError(f"Unknown embedding model keys: {unknown}")
    return selected


def validate_experiment_mode(experiment_mode: str) -> str:
    if experiment_mode not in {"local", "global"}:
        raise ValueError("experiment_mode must be 'local' or 'global'.")
    return experiment_mode


def similarity_output_path(
    model_key: str,
    experiment_mode: str,
    record_limit: int | None,
) -> Path:
    validate_experiment_mode(experiment_mode)
    if record_limit is not None and int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    output_stem = model_key if record_limit is None else f"{model_key}.limit-{int(record_limit)}"
    return OUTPUT_DIR / experiment_mode / embedding_precision_mode() / f"{output_stem}.json"


print(f"local_dataset={LOCAL_DATASET_PATH}")
print(f"output_dir={OUTPUT_DIR}")

## Embedding Backends

In [ ]:
def embedding_device() -> str:
    return f"cuda:{EMBEDDING_GPU_ID}" if torch.cuda.is_available() else "cpu"


def embedding_compute_dtype() -> str:
    return "float16" if EMBEDDING_USE_FP16 and torch.cuda.is_available() else "float32"


def embedding_precision_mode() -> str:
    return "fp16" if embedding_compute_dtype() == "float16" else "fp32"


def load_embedding_model(model_key: str):
    spec = EMBEDDING_MODEL_SPECS[model_key]
    device = embedding_device()
    if spec["backend"] == "sentence_transformers":
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(spec["repo"], device=device)
        model.max_seq_length = min(int(model.max_seq_length), EMBEDDING_MAX_LENGTH)
        if embedding_compute_dtype() == "float16":
            model.half()
    elif spec["backend"] == "flag_embedding":
        from FlagEmbedding import BGEM3FlagModel

        model = BGEM3FlagModel(
            spec["repo"],
            use_fp16=embedding_compute_dtype() == "float16",
            devices=device,
        )
    else:
        raise ValueError(f"Unsupported embedding backend: {spec['backend']}")
    print(
        f"loaded_embedding_model={model_key} repo={spec['repo']} "
        f"backend={spec['backend']} device={device} "
        f"compute_dtype={embedding_compute_dtype()}"
    )
    return model


def embedding_effective_max_length(model, model_key: str) -> int:
    if EMBEDDING_MODEL_SPECS[model_key]["backend"] == "sentence_transformers":
        return int(model.max_seq_length)
    return EMBEDDING_MAX_LENGTH


def format_embedding_prompts(
    model_key: str,
    prompts: list[str],
    prompt_role: str,
) -> list[str]:
    prefixes = EMBEDDING_MODEL_SPECS[model_key]["prefixes"]
    if prompt_role not in prefixes:
        raise ValueError(f"Unsupported prompt role: {prompt_role}")
    prefix = prefixes[prompt_role]
    return [f"{prefix}{prompt}" for prompt in prompts]


def encode_flag_embedding_prompts(model, prompts: list[str]) -> dict[str, Any]:
    with redirect_stderr(StringIO()):
        return model.encode(
            prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            max_length=EMBEDDING_MAX_LENGTH,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )


def encode_prompts(
    model,
    model_key: str,
    prompts: list[str],
    prompt_role: str,
) -> np.ndarray:
    if not prompts:
        raise ValueError("Cannot encode an empty prompt list.")
    spec = EMBEDDING_MODEL_SPECS[model_key]
    formatted_prompts = format_embedding_prompts(model_key, prompts, prompt_role)
    if spec["backend"] == "sentence_transformers":
        embeddings = model.encode(
            formatted_prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    else:
        encoded = encode_flag_embedding_prompts(model, formatted_prompts)
        embeddings = encoded["dense_vecs"]

    embeddings = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=-1, keepdims=True)
    embeddings = embeddings / np.clip(norms, 1e-12, None)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(prompts):
        raise ValueError(
            f"Unexpected embedding shape for {model_key}: {embeddings.shape}; "
            f"expected ({len(prompts)}, hidden_size)."
        )
    return embeddings


def move_embedding_model_to_cpu(model, model_key: str) -> None:
    backend = EMBEDDING_MODEL_SPECS[model_key]["backend"]
    try:
        if backend == "sentence_transformers":
            model.to("cpu")
        else:
            underlying_model = getattr(model, "model", None)
            if underlying_model is not None and hasattr(underlying_model, "to"):
                underlying_model.to("cpu")
    except Exception as error:
        print(f"embedding_model_cpu_move_warning={model_key}: {error}")


def clear_embedding_memory(model_key: str) -> None:
    gc.collect()
    if not torch.cuda.is_available():
        return
    try:
        torch.cuda.synchronize(EMBEDDING_GPU_ID)
    except Exception:
        pass
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    allocated = torch.cuda.memory_allocated(EMBEDDING_GPU_ID) / 1024**3
    reserved = torch.cuda.memory_reserved(EMBEDDING_GPU_ID) / 1024**3
    print(
        f"released_embedding_model={model_key} "
        f"cuda_allocated_gb={allocated:.3f} cuda_reserved_gb={reserved:.3f}"
    )

## Record Construction

Local mode preserves complete within-record prompt comparisons. Global mode constructs fixed positive and locality query cohorts for each repeat, then grows a nested edit bank around the same query owners so bank-size changes measure added retrieval competition rather than different sampled prompts.

In [ ]:
PAIR_AGGREGATION = "all_prompt_pairs"
PAIR_SPECS = (
    ("p1-p2", "edit", "generality", "edit - generality"),
    ("p1-p3", "edit", "locality", "edit - locality"),
    ("p2-p3", "generality", "locality", "generality - locality"),
)


def load_similarity_dataset(dataset_source: str):
    if dataset_source == "local":
        return load_from_disk(str(LOCAL_DATASET_PATH))
    if dataset_source == "huggingface":
        return load_dataset(DATASET_REPO, DATASET_CONFIG, split=DATASET_SPLIT)
    raise ValueError("dataset_source must be 'local' or 'huggingface'.")


def source_record_count(dataset, record_limit: int | None) -> int:
    if record_limit is None:
        return len(dataset)
    if int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    return min(int(record_limit), len(dataset))


def build_local_similarity_records(
    dataset,
    record_limit: int | None,
) -> list[dict[str, Any]]:
    records = []
    for source_index in range(source_record_count(dataset, record_limit)):
        source_record = dataset[source_index]
        role_prompts = {
            "edit": [source_record["edit"]["prompt"]],
            "generality": list(source_record["generality"]),
            "locality": [item["prompt"] for item in source_record["locality"]],
        }
        for role, prompts in role_prompts.items():
            if not prompts or any(not prompt for prompt in prompts):
                raise ValueError(f"Empty {role} prompt list at source index {source_index}.")
        records.append({"source_index": source_index, "role_prompts": role_prompts})
    return records


def build_global_similarity_records(
    dataset,
    record_limit: int | None,
) -> list[dict[str, Any]]:
    records = []
    for source_index in range(source_record_count(dataset, record_limit)):
        source_record = dataset[source_index]
        edit_prompt = source_record["edit"]["prompt"]
        generality_prompts = list(source_record["generality"])
        locality_prompts = [item["prompt"] for item in source_record["locality"]]
        if not edit_prompt or not generality_prompts or not locality_prompts:
            raise ValueError(f"Incomplete retrieval record at source index {source_index}.")
        records.append({
            "source_index": source_index,
            "edit_prompt": edit_prompt,
            "generality_prompts": generality_prompts,
            "locality_prompts": locality_prompts,
        })
    return records


def prompt_key(prompt: str) -> str:
    normalized = unicodedata.normalize("NFKC", prompt).casefold()
    characters = [
        character
        if character.isalnum() or unicodedata.category(character).startswith("S")
        else " "
        for character in normalized
    ]
    return " ".join("".join(characters).split())


def seeded_rng(random_seed: int, *parts) -> np.random.Generator:
    payload = "|".join(str(part) for part in (random_seed, *parts)).encode("utf-8")
    seed = int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")
    return np.random.default_rng(seed)


def sample_values(values: list, limit: int | None, rng: np.random.Generator) -> list:
    if limit is None or len(values) <= limit:
        return values
    indices = np.sort(rng.choice(len(values), size=limit, replace=False))
    return [values[int(index)] for index in indices]


def resolve_global_bank_sizes(
    bank_sizes: list[int | None],
    universe_size: int,
) -> list[tuple[int | None, int]]:
    resolved = []
    seen = set()
    for requested_size in bank_sizes:
        actual_size = universe_size if requested_size is None else int(requested_size)
        if actual_size < 2:
            raise ValueError("Every global bank must contain at least two edits.")
        if actual_size > universe_size:
            print(f"skipped_bank_size={actual_size} universe_size={universe_size}")
            continue
        key = (requested_size is None, actual_size)
        if key not in seen:
            seen.add(key)
            resolved.append((requested_size, actual_size))
    if not resolved:
        raise ValueError("No configured global bank size fits the dataset.")
    return resolved

## Similarity Calculation

Global mode reuses the same queries while adding distractor edits. It records raw retrieval, score/margin separation, the legacy threshold calibrated only for 95% positive activation, locality-controlled score gates, and locality-controlled score-plus-margin gates. This preserves the meaning of the previous global result while extending it into a deployable routing test.

In [ ]:
def analyze_local_record_batch(
    record_batch: list[dict[str, Any]],
    model,
    model_key: str,
) -> list[dict[str, Any]]:
    prompts = [
        prompt
        for record in record_batch
        for role_prompts in record["role_prompts"].values()
        for prompt in role_prompts
    ]
    embeddings = encode_prompts(model, model_key, prompts, prompt_role="symmetric")

    analyzed_records = []
    offset = 0
    for record in record_batch:
        role_embeddings = {}
        prompt_counts = {}
        for role, role_prompts in record["role_prompts"].items():
            count = len(role_prompts)
            role_embeddings[role] = embeddings[offset : offset + count]
            prompt_counts[role] = count
            offset += count

        pair_values = {}
        for pair_key, left_role, right_role, _ in PAIR_SPECS:
            matrix = role_embeddings[left_role] @ role_embeddings[right_role].T
            pair_values[pair_key] = [
                round(float(value), 8) for value in matrix.reshape(-1)
            ]
        analyzed_records.append({
            "source_index": record["source_index"],
            "prompt_counts": prompt_counts,
            "pair_values": pair_values,
        })
    return analyzed_records


def aggregate_local_records(records: list[dict[str, Any]]) -> dict[str, Any]:
    if not records:
        raise ValueError("No records were analyzed.")

    pairs = {}
    for pair_key, _, _, label in PAIR_SPECS:
        values = np.asarray(
            [
                value
                for record in records
                for value in record["pair_values"][pair_key]
            ],
            dtype=np.float64,
        )
        if values.size == 0:
            raise ValueError(f"No values were produced for pair {pair_key}.")
        pairs[pair_key] = {
            "label": label,
            "count": int(values.size),
            "mean": round(float(values.mean()), 8),
            "std": round(float(values.std(ddof=0)), 8),
        }
    return {"record_count": len(records), "pairs": pairs}


def write_json_atomic(payload: dict, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(f"{destination.suffix}.tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=True, allow_nan=False)
    temporary.replace(destination)


def similarity_dataset_metadata(dataset, dataset_source: str) -> dict[str, Any]:
    metadata = {
        "source": dataset_source,
        "rows": len(dataset),
        "fingerprint": getattr(dataset, "_fingerprint", None),
    }
    if dataset_source == "local":
        metadata["dataset_id"] = "uniedit/final"
    else:
        metadata.update({
            "repo": DATASET_REPO,
            "config": DATASET_CONFIG,
            "split": DATASET_SPLIT,
        })
    return metadata


def run_local_similarity_analysis(
    dataset,
    records: list[dict[str, Any]],
    model,
    model_key: str,
    output_path: Path,
    record_batch_size: int,
    dataset_metadata: dict[str, Any],
    experiment: dict[str, Any],
) -> None:
    if record_batch_size <= 0:
        raise ValueError("record_batch_size must be positive.")

    completed_records = []
    for start in tqdm(
        range(0, len(records), record_batch_size),
        desc=f"Local embeddings [{model_key}]",
    ):
        completed_records.extend(
            analyze_local_record_batch(
                records[start : start + record_batch_size],
                model=model,
                model_key=model_key,
            )
        )

    completed_records.sort(key=lambda record: record["source_index"])
    result = build_result_header(
        model=model,
        model_key=model_key,
        dataset_metadata=dataset_metadata,
        experiment=experiment,
    )
    result.update({
        "pair_aggregation": PAIR_AGGREGATION,
        "pair_order": [
            {"pair": pair_key, "label": label}
            for pair_key, _, _, label in PAIR_SPECS
        ],
        "records": completed_records,
        "aggregate": aggregate_local_records(completed_records),
    })
    write_json_atomic(result, output_path)
    print(f"saved={output_path} records={len(completed_records)}")


GLOBAL_RETRIEVAL_METRICS = (
    "recall_at_1",
    "recall_at_5",
    "recall_at_10",
    "mrr",
    "correct_margin_mean",
    "correct_margin_std",
    "top1_margin_mean",
)
GLOBAL_GATE_METRICS = (
    "score_threshold",
    "margin_threshold",
    "calibration_correct_routed_rate",
    "calibration_locality_false_activation_rate",
    "positive_activation_rate",
    "positive_rejection_rate",
    "correct_routed_rate",
    "incorrect_routed_rate",
    "locality_false_activation_rate",
)
GLOBAL_GATE_ORDER = ("score", "score_margin")
GLOBAL_GATE_LABELS = {
    "score": "top score",
    "score_margin": "top score + top1/top2 margin",
}
GLOBAL_SEPARATION_METRICS = (
    "score_auc",
    "margin_auc",
    "positive_score_mean",
    "positive_score_p05",
    "positive_score_p50",
    "positive_score_p95",
    "locality_score_mean",
    "locality_score_p05",
    "locality_score_p50",
    "locality_score_p95",
    "score_conservative_gap",
    "positive_margin_mean",
    "positive_margin_p05",
    "positive_margin_p50",
    "positive_margin_p95",
    "locality_margin_mean",
    "locality_margin_p05",
    "locality_margin_p50",
    "locality_margin_p95",
    "margin_conservative_gap",
)
GLOBAL_LEGACY_GATE_METRICS = (
    "score_threshold",
    "calibration_positive_activation_rate",
    "calibration_correct_routed_rate",
    "calibration_locality_false_activation_rate",
    "positive_activation_rate",
    "positive_rejection_rate",
    "correct_routed_rate",
    "incorrect_routed_rate",
    "locality_false_activation_rate",
)


def exact_positive_retrieval(
    query_embeddings: np.ndarray,
    bank: torch.Tensor,
    owner_positions: np.ndarray,
    retrieval_batch_size: int,
) -> dict[str, np.ndarray]:
    if len(query_embeddings) != len(owner_positions):
        raise ValueError("Each positive query must have one owner position.")
    device = bank.device
    outputs = {
        "ranks": [],
        "top1_scores": [],
        "top2_scores": [],
        "top1_positions": [],
        "top2_positions": [],
        "correct_scores": [],
        "best_negative_scores": [],
        "top1_correct": [],
    }
    with torch.inference_mode():
        for start in range(0, len(query_embeddings), retrieval_batch_size):
            stop = min(start + retrieval_batch_size, len(query_embeddings))
            queries = torch.as_tensor(
                query_embeddings[start:stop],
                dtype=torch.float32,
                device=device,
            )
            owners = torch.as_tensor(
                owner_positions[start:stop],
                dtype=torch.long,
                device=device,
            )
            scores = queries @ bank.T
            rows = torch.arange(len(owners), device=device)
            correct = scores[rows, owners]
            top_scores, top_positions = torch.topk(scores, k=2, dim=1)
            top1_correct = top_positions[:, 0].eq(owners)
            best_negative = torch.where(
                top1_correct,
                top_scores[:, 1],
                top_scores[:, 0],
            )
            outputs["ranks"].append(
                (scores > correct[:, None]).sum(dim=1).add(1).cpu().numpy()
            )
            outputs["top1_scores"].append(top_scores[:, 0].cpu().numpy())
            outputs["top2_scores"].append(top_scores[:, 1].cpu().numpy())
            outputs["top1_positions"].append(top_positions[:, 0].cpu().numpy())
            outputs["top2_positions"].append(top_positions[:, 1].cpu().numpy())
            outputs["correct_scores"].append(correct.cpu().numpy())
            outputs["best_negative_scores"].append(best_negative.cpu().numpy())
            outputs["top1_correct"].append(top1_correct.cpu().numpy())
            del queries, owners, scores, correct, top_scores, top_positions, best_negative
    result = {key: np.concatenate(values) for key, values in outputs.items()}
    result["top1_margins"] = result["top1_scores"] - result["top2_scores"]
    return result


def exact_locality_retrieval(
    query_embeddings: np.ndarray,
    bank: torch.Tensor,
    retrieval_batch_size: int,
) -> dict[str, np.ndarray]:
    device = bank.device
    outputs = {
        "top1_scores": [],
        "top2_scores": [],
        "top1_positions": [],
        "top2_positions": [],
    }
    with torch.inference_mode():
        for start in range(0, len(query_embeddings), retrieval_batch_size):
            queries = torch.as_tensor(
                query_embeddings[start : start + retrieval_batch_size],
                dtype=torch.float32,
                device=device,
            )
            scores = queries @ bank.T
            top_scores, top_positions = torch.topk(scores, k=2, dim=1)
            outputs["top1_scores"].append(top_scores[:, 0].cpu().numpy())
            outputs["top2_scores"].append(top_scores[:, 1].cpu().numpy())
            outputs["top1_positions"].append(top_positions[:, 0].cpu().numpy())
            outputs["top2_positions"].append(top_positions[:, 1].cpu().numpy())
            del queries, scores, top_scores, top_positions
    result = {key: np.concatenate(values) for key, values in outputs.items()}
    result["top1_margins"] = result["top1_scores"] - result["top2_scores"]
    return result


def far_target_key(target: float) -> str:
    return f"{target:.6f}".rstrip("0").rstrip(".")


def maximum_false_activations(query_count: int, target: float) -> int:
    return int(np.floor(query_count * target + 1e-12))


def score_threshold_for_budget(
    scores: np.ndarray,
    eligible: np.ndarray,
    maximum_active: int,
) -> float:
    eligible_scores = scores[eligible]
    if eligible_scores.size <= maximum_active:
        return float(np.nextafter(-1.0, -np.inf))
    ordered = np.sort(eligible_scores)[::-1]
    return float(np.nextafter(ordered[maximum_active], np.inf))


def gate_active_mask(
    retrieval: dict[str, np.ndarray],
    score_threshold: float,
    margin_threshold: float,
) -> np.ndarray:
    return (
        (retrieval["top1_scores"] >= score_threshold)
        & (retrieval["top1_margins"] >= margin_threshold)
    )


def evaluate_gate(
    positive: dict[str, np.ndarray],
    locality: dict[str, np.ndarray],
    score_threshold: float,
    margin_threshold: float,
) -> dict[str, float]:
    positive_active = gate_active_mask(positive, score_threshold, margin_threshold)
    locality_active = gate_active_mask(locality, score_threshold, margin_threshold)
    top1_correct = positive["top1_correct"]
    return {
        "positive_activation_rate": float(positive_active.mean()),
        "positive_rejection_rate": float((~positive_active).mean()),
        "correct_routed_rate": float(np.mean(positive_active & top1_correct)),
        "incorrect_routed_rate": float(np.mean(positive_active & ~top1_correct)),
        "locality_false_activation_rate": float(locality_active.mean()),
    }


def calibrate_score_gate(
    _positive: dict[str, np.ndarray],
    locality: dict[str, np.ndarray],
    target: float,
) -> tuple[float, float]:
    maximum_active = maximum_false_activations(len(locality["top1_scores"]), target)
    threshold = score_threshold_for_budget(
        locality["top1_scores"],
        np.ones(len(locality["top1_scores"]), dtype=bool),
        maximum_active,
    )
    return threshold, 0.0


def calibrate_score_margin_gate(
    positive: dict[str, np.ndarray],
    locality: dict[str, np.ndarray],
    target: float,
) -> tuple[float, float]:
    maximum_active = maximum_false_activations(len(locality["top1_scores"]), target)
    margin_candidates = np.concatenate((
        np.asarray([0.0]),
        np.nextafter(np.unique(locality["top1_margins"]), np.inf),
    ))
    best_key = None
    best_thresholds = None
    for margin_threshold in margin_candidates:
        locality_eligible = locality["top1_margins"] >= margin_threshold
        score_threshold = score_threshold_for_budget(
            locality["top1_scores"],
            locality_eligible,
            maximum_active,
        )
        positive_active = gate_active_mask(
            positive,
            score_threshold,
            float(margin_threshold),
        )
        locality_active = gate_active_mask(
            locality,
            score_threshold,
            float(margin_threshold),
        )
        correct_count = int(np.count_nonzero(
            positive_active & positive["top1_correct"]
        ))
        false_count = int(np.count_nonzero(locality_active))
        selection_key = (
            correct_count,
            -false_count,
            -float(margin_threshold),
            -score_threshold,
        )
        if best_key is None or selection_key > best_key:
            best_key = selection_key
            best_thresholds = (score_threshold, float(margin_threshold))
    return best_thresholds



def pairwise_auc(positive: np.ndarray, negative: np.ndarray) -> float:
    ordered_negative = np.sort(negative)
    strict_wins = np.searchsorted(ordered_negative, positive, side="left")
    weak_wins = np.searchsorted(ordered_negative, positive, side="right")
    return float(np.mean(
        (strict_wins + 0.5 * (weak_wins - strict_wins)) / len(negative)
    ))


def distribution_statistics(values: np.ndarray, prefix: str) -> dict[str, float]:
    return {
        f"{prefix}_mean": round(float(values.mean()), 8),
        f"{prefix}_p05": round(float(np.quantile(values, 0.05)), 8),
        f"{prefix}_p50": round(float(np.quantile(values, 0.50)), 8),
        f"{prefix}_p95": round(float(np.quantile(values, 0.95)), 8),
    }


def score_separation_metrics(
    positive: dict[str, np.ndarray],
    locality: dict[str, np.ndarray],
) -> dict[str, float]:
    positive_scores = positive["top1_scores"]
    locality_scores = locality["top1_scores"]
    positive_margins = positive["top1_margins"]
    locality_margins = locality["top1_margins"]
    positive_score_p05 = float(np.quantile(positive_scores, 0.05))
    locality_score_p95 = float(np.quantile(locality_scores, 0.95))
    positive_margin_p05 = float(np.quantile(positive_margins, 0.05))
    locality_margin_p95 = float(np.quantile(locality_margins, 0.95))
    return {
        "score_auc": round(pairwise_auc(positive_scores, locality_scores), 8),
        "margin_auc": round(pairwise_auc(positive_margins, locality_margins), 8),
        **distribution_statistics(positive_scores, "positive_score"),
        **distribution_statistics(locality_scores, "locality_score"),
        "score_conservative_gap": round(
            positive_score_p05 - locality_score_p95, 8
        ),
        **distribution_statistics(positive_margins, "positive_margin"),
        **distribution_statistics(locality_margins, "locality_margin"),
        "margin_conservative_gap": round(
            positive_margin_p05 - locality_margin_p95, 8
        ),
    }


def calibrate_legacy_positive_gate(
    calibration_positive: dict[str, np.ndarray],
    calibration_locality: dict[str, np.ndarray],
    evaluation_positive: dict[str, np.ndarray],
    evaluation_locality: dict[str, np.ndarray],
    target_positive_activation: float,
) -> dict[str, float]:
    score_threshold = float(np.quantile(
        calibration_positive["top1_scores"],
        1.0 - target_positive_activation,
        method="lower",
    ))
    calibration_metrics = evaluate_gate(
        calibration_positive,
        calibration_locality,
        score_threshold,
        0.0,
    )
    evaluation_metrics = evaluate_gate(
        evaluation_positive,
        evaluation_locality,
        score_threshold,
        0.0,
    )
    return {
        "score_threshold": score_threshold,
        "calibration_positive_activation_rate": round(
            calibration_metrics["positive_activation_rate"], 8
        ),
        "calibration_correct_routed_rate": round(
            calibration_metrics["correct_routed_rate"], 8
        ),
        "calibration_locality_false_activation_rate": round(
            calibration_metrics["locality_false_activation_rate"], 8
        ),
        **{
            key: round(value, 8)
            for key, value in evaluation_metrics.items()
        },
    }


def calibrate_global_gates(
    calibration_positive: dict[str, np.ndarray],
    calibration_locality: dict[str, np.ndarray],
    evaluation_positive: dict[str, np.ndarray],
    evaluation_locality: dict[str, np.ndarray],
    far_targets: list[float],
) -> dict[str, dict[str, dict[str, float]]]:
    gates = {gate_key: {} for gate_key in GLOBAL_GATE_ORDER}
    for target in far_targets:
        target_key = far_target_key(target)
        calibrators = {
            "score": calibrate_score_gate,
            "score_margin": calibrate_score_margin_gate,
        }
        for gate_key, calibrator in calibrators.items():
            score_threshold, margin_threshold = calibrator(
                calibration_positive,
                calibration_locality,
                target,
            )
            calibration_metrics = evaluate_gate(
                calibration_positive,
                calibration_locality,
                score_threshold,
                margin_threshold,
            )
            evaluation_metrics = evaluate_gate(
                evaluation_positive,
                evaluation_locality,
                score_threshold,
                margin_threshold,
            )
            gates[gate_key][target_key] = {
                "target_locality_far": target,
                "score_threshold": float(score_threshold),
                "margin_threshold": float(margin_threshold),
                "calibration_correct_routed_rate": round(
                    calibration_metrics["correct_routed_rate"], 8
                ),
                "calibration_locality_false_activation_rate": round(
                    calibration_metrics["locality_false_activation_rate"], 8
                ),
                **{
                    key: round(value, 8)
                    for key, value in evaluation_metrics.items()
                },
            }
    return gates


def unique_locality_queries(
    records: list[dict[str, Any]],
    source_indices: np.ndarray,
) -> list[dict[str, Any]]:
    locality_by_key = {}
    for source_index in source_indices:
        for prompt in records[int(source_index)]["locality_prompts"]:
            locality_by_key.setdefault(
                prompt_key(prompt),
                {"prompt": prompt, "source_index": int(source_index)},
            )
    return list(locality_by_key.values())


def build_fixed_global_cohort(
    records: list[dict[str, Any]],
    settings: dict[str, Any],
    repeat_index: int,
    minimum_bank_size: int,
) -> dict[str, Any]:
    owner_count = settings["query_owner_count"]
    if owner_count > minimum_bank_size:
        raise ValueError(
            "global_query_owner_count cannot exceed the smallest global bank size."
        )
    if len(records) - owner_count < 2:
        raise ValueError("Global routing requires records outside the positive owner set.")

    repeat_rng = seeded_rng(settings["random_seed"], "cohort", repeat_index)
    query_rng = seeded_rng(settings["random_seed"], "query_sample", repeat_index)
    record_order = repeat_rng.permutation(len(records)).astype(np.int64)
    owner_indices = record_order[:owner_count]
    distractor_indices = record_order[owner_count:]

    owner_positions = repeat_rng.permutation(owner_count)
    calibration_owner_count = int(round(
        owner_count * settings["calibration_fraction"]
    ))
    calibration_owner_count = min(
        max(calibration_owner_count, 1), owner_count - 1
    )
    calibration_positions = set(
        int(position) for position in owner_positions[:calibration_owner_count]
    )

    calibration_positive = []
    evaluation_positive = []
    for owner_position, source_index in enumerate(owner_indices):
        destination = (
            calibration_positive
            if owner_position in calibration_positions
            else evaluation_positive
        )
        destination.extend({
            "prompt": prompt,
            "owner_position": owner_position,
            "source_index": int(source_index),
        } for prompt in records[int(source_index)]["generality_prompts"])
    calibration_positive = sample_values(
        calibration_positive,
        settings["positive_calibration_query_limit"],
        query_rng,
    )
    evaluation_positive = sample_values(
        evaluation_positive,
        settings["positive_evaluation_query_limit"],
        query_rng,
    )

    locality_source_split = int(round(
        len(distractor_indices) * settings["calibration_fraction"]
    ))
    locality_source_split = min(
        max(locality_source_split, 1), len(distractor_indices) - 1
    )
    calibration_locality_pool = unique_locality_queries(
        records,
        distractor_indices[:locality_source_split],
    )
    calibration_locality_keys = {
        prompt_key(query["prompt"])
        for query in calibration_locality_pool
    }
    evaluation_locality_pool = [
        query
        for query in unique_locality_queries(
            records,
            distractor_indices[locality_source_split:],
        )
        if prompt_key(query["prompt"]) not in calibration_locality_keys
    ]
    calibration_locality = sample_values(
        calibration_locality_pool,
        settings["locality_calibration_query_limit"],
        query_rng,
    )
    evaluation_locality = sample_values(
        evaluation_locality_pool,
        settings["locality_evaluation_query_limit"],
        query_rng,
    )
    if not all((
        calibration_positive,
        evaluation_positive,
        calibration_locality,
        evaluation_locality,
    )):
        raise ValueError(
            f"Insufficient fixed global queries for repeat={repeat_index}."
        )
    return {
        "owner_indices": owner_indices,
        "distractor_indices": distractor_indices,
        "calibration_owner_count": calibration_owner_count,
        "calibration_positive": calibration_positive,
        "evaluation_positive": evaluation_positive,
        "calibration_locality": calibration_locality,
        "evaluation_locality": evaluation_locality,
    }


def fixed_cohort_fingerprint(cohort: dict[str, Any]) -> str:
    digest = hashlib.sha256()
    for group_name in (
        "calibration_positive",
        "evaluation_positive",
        "calibration_locality",
        "evaluation_locality",
    ):
        digest.update(group_name.encode("utf-8"))
        for query in cohort[group_name]:
            digest.update(prompt_key(query["prompt"]).encode("utf-8"))
            digest.update(str(query["source_index"]).encode("ascii"))
    return digest.hexdigest()


def retrieval_metrics(retrieval: dict[str, np.ndarray]) -> dict[str, float]:
    margins = retrieval["correct_scores"] - retrieval["best_negative_scores"]
    return {
        "recall_at_1": round(float(np.mean(retrieval["ranks"] <= 1)), 8),
        "recall_at_5": round(float(np.mean(retrieval["ranks"] <= 5)), 8),
        "recall_at_10": round(float(np.mean(retrieval["ranks"] <= 10)), 8),
        "mrr": round(float(np.mean(1.0 / retrieval["ranks"])), 8),
        "correct_margin_mean": round(float(margins.mean()), 8),
        "correct_margin_std": round(float(margins.std(ddof=0)), 8),
        "top1_margin_mean": round(float(retrieval["top1_margins"].mean()), 8),
    }


def diagnostic_positions(
    mask: np.ndarray,
    order_values: np.ndarray,
    limit: int,
) -> np.ndarray:
    positions = np.flatnonzero(mask)
    if not limit or not len(positions):
        return np.asarray([], dtype=np.int64)
    order = np.argsort(order_values[positions])[::-1]
    return positions[order[:limit]]


def positive_diagnostics(
    queries: list[dict[str, Any]],
    retrieval: dict[str, np.ndarray],
    bank_indices: np.ndarray,
    records: list[dict[str, Any]],
    positions: np.ndarray,
) -> list[dict[str, Any]]:
    diagnostics = []
    for query_position in positions:
        query = queries[int(query_position)]
        top1_position = int(retrieval["top1_positions"][query_position])
        top2_position = int(retrieval["top2_positions"][query_position])
        top1_source = int(bank_indices[top1_position])
        top2_source = int(bank_indices[top2_position])
        diagnostics.append({
            "query": query["prompt"],
            "expected_source_index": query["source_index"],
            "expected_edit": records[query["source_index"]]["edit_prompt"],
            "retrieved_source_index": top1_source,
            "retrieved_edit": records[top1_source]["edit_prompt"],
            "second_source_index": top2_source,
            "second_edit": records[top2_source]["edit_prompt"],
            "rank": int(retrieval["ranks"][query_position]),
            "top1_score": round(float(retrieval["top1_scores"][query_position]), 8),
            "top2_score": round(float(retrieval["top2_scores"][query_position]), 8),
            "top1_margin": round(float(retrieval["top1_margins"][query_position]), 8),
            "correct_score": round(float(retrieval["correct_scores"][query_position]), 8),
        })
    return diagnostics


def locality_diagnostics(
    queries: list[dict[str, Any]],
    retrieval: dict[str, np.ndarray],
    bank_indices: np.ndarray,
    records: list[dict[str, Any]],
    positions: np.ndarray,
) -> list[dict[str, Any]]:
    diagnostics = []
    for query_position in positions:
        top1_position = int(retrieval["top1_positions"][query_position])
        top2_position = int(retrieval["top2_positions"][query_position])
        top1_source = int(bank_indices[top1_position])
        top2_source = int(bank_indices[top2_position])
        diagnostics.append({
            "query": queries[int(query_position)]["prompt"],
            "retrieved_source_index": top1_source,
            "retrieved_edit": records[top1_source]["edit_prompt"],
            "second_source_index": top2_source,
            "second_edit": records[top2_source]["edit_prompt"],
            "top1_score": round(float(retrieval["top1_scores"][query_position]), 8),
            "top2_score": round(float(retrieval["top2_scores"][query_position]), 8),
            "top1_margin": round(float(retrieval["top1_margins"][query_position]), 8),
        })
    return diagnostics


def routing_diagnostics(
    evaluation_queries: list[dict[str, Any]],
    locality_queries: list[dict[str, Any]],
    evaluation: dict[str, np.ndarray],
    locality: dict[str, np.ndarray],
    gates: dict[str, dict[str, dict[str, float]]],
    bank_indices: np.ndarray,
    records: list[dict[str, Any]],
    limit: int,
) -> dict[str, Any]:
    wrong_positions = diagnostic_positions(
        ~evaluation["top1_correct"],
        evaluation["top1_scores"],
        limit,
    )
    diagnostics = {
        "wrong_top1": positive_diagnostics(
            evaluation_queries,
            evaluation,
            bank_indices,
            records,
            wrong_positions,
        ),
        "gates": {},
    }
    for gate_key, targets in gates.items():
        diagnostics["gates"][gate_key] = {}
        for target_key, gate in targets.items():
            positive_active = gate_active_mask(
                evaluation,
                gate["score_threshold"],
                gate["margin_threshold"],
            )
            locality_active = gate_active_mask(
                locality,
                gate["score_threshold"],
                gate["margin_threshold"],
            )
            rejected_positions = diagnostic_positions(
                evaluation["top1_correct"] & ~positive_active,
                evaluation["top1_scores"],
                limit,
            )
            false_positions = diagnostic_positions(
                locality_active,
                locality["top1_scores"],
                limit,
            )
            diagnostics["gates"][gate_key][target_key] = {
                "rejected_positive": positive_diagnostics(
                    evaluation_queries,
                    evaluation,
                    bank_indices,
                    records,
                    rejected_positions,
                ),
                "false_locality": locality_diagnostics(
                    locality_queries,
                    locality,
                    bank_indices,
                    records,
                    false_positions,
                ),
            }
    return diagnostics


def aggregate_metrics(
    repeats: list[dict[str, Any]],
    metric_path: tuple[str, ...],
    metric_names: tuple[str, ...],
) -> dict[str, Any]:
    aggregate = {}
    for metric_name in metric_names:
        values = []
        for repeat in repeats:
            source = repeat
            for path_part in metric_path:
                source = source[path_part]
            values.append(source[metric_name])
        values = np.asarray(values, dtype=np.float64)
        aggregate[metric_name] = {
            "mean": round(float(values.mean()), 8),
            "std": round(float(values.std(ddof=0)), 8),
        }
    return aggregate


def aggregate_global_repeats(repeats: list[dict[str, Any]]) -> dict[str, Any]:
    aggregate = {
        "retrieval": aggregate_metrics(
            repeats,
            ("retrieval",),
            GLOBAL_RETRIEVAL_METRICS,
        ),
        "separation": aggregate_metrics(
            repeats,
            ("separation",),
            GLOBAL_SEPARATION_METRICS,
        ),
        "legacy_gate": aggregate_metrics(
            repeats,
            ("legacy_gate",),
            GLOBAL_LEGACY_GATE_METRICS,
        ),
        "gates": {},
    }
    for gate_key in GLOBAL_GATE_ORDER:
        aggregate["gates"][gate_key] = {}
        for target_key in repeats[0]["gates"][gate_key]:
            aggregate["gates"][gate_key][target_key] = aggregate_metrics(
                repeats,
                ("gates", gate_key, target_key),
                GLOBAL_GATE_METRICS,
            )
    return aggregate


def run_global_similarity_analysis(
    records: list[dict[str, Any]],
    model,
    model_key: str,
    output_path: Path,
    dataset_metadata: dict[str, Any],
    experiment: dict[str, Any],
) -> None:
    settings = experiment["settings"]
    edit_embeddings = encode_prompts(
        model,
        model_key,
        [record["edit_prompt"] for record in records],
        prompt_role="passage",
    )
    resolved_sizes = resolve_global_bank_sizes(settings["bank_sizes"], len(records))
    minimum_bank_size = min(bank_size for _, bank_size in resolved_sizes)
    repeat_results_by_size = {
        bank_size: [] for _, bank_size in resolved_sizes
    }
    progress = tqdm(
        total=len(resolved_sizes) * settings["repeats"],
        desc=f"Global routing [{model_key}]",
    )
    for repeat_index in range(settings["repeats"]):
        cohort = build_fixed_global_cohort(
            records,
            settings,
            repeat_index,
            minimum_bank_size,
        )
        query_groups = (
            cohort["calibration_positive"],
            cohort["evaluation_positive"],
            cohort["calibration_locality"],
            cohort["evaluation_locality"],
        )
        all_queries = [
            query["prompt"]
            for query_group in query_groups
            for query in query_group
        ]
        query_embeddings = encode_prompts(
            model,
            model_key,
            all_queries,
            prompt_role="query",
        )
        group_stops = np.cumsum([len(group) for group in query_groups])
        calibration_positive_embeddings = query_embeddings[:group_stops[0]]
        evaluation_positive_embeddings = query_embeddings[
            group_stops[0] : group_stops[1]
        ]
        calibration_locality_embeddings = query_embeddings[
            group_stops[1] : group_stops[2]
        ]
        evaluation_locality_embeddings = query_embeddings[
            group_stops[2] : group_stops[3]
        ]
        calibration_owners = np.asarray([
            query["owner_position"]
            for query in cohort["calibration_positive"]
        ])
        evaluation_owners = np.asarray([
            query["owner_position"]
            for query in cohort["evaluation_positive"]
        ])

        cohort_fingerprint = fixed_cohort_fingerprint(cohort)
        for _requested_size, bank_size in resolved_sizes:
            distractor_count = bank_size - len(cohort["owner_indices"])
            bank_indices = np.concatenate((
                cohort["owner_indices"],
                cohort["distractor_indices"][:distractor_count],
            ))
            retrieval_bank = torch.as_tensor(
                edit_embeddings[bank_indices],
                dtype=torch.float32,
                device=torch.device(embedding_device()),
            )
            calibration_positive = exact_positive_retrieval(
                calibration_positive_embeddings,
                retrieval_bank,
                calibration_owners,
                settings["retrieval_batch_size"],
            )
            evaluation_positive = exact_positive_retrieval(
                evaluation_positive_embeddings,
                retrieval_bank,
                evaluation_owners,
                settings["retrieval_batch_size"],
            )
            calibration_locality = exact_locality_retrieval(
                calibration_locality_embeddings,
                retrieval_bank,
                settings["retrieval_batch_size"],
            )
            evaluation_locality = exact_locality_retrieval(
                evaluation_locality_embeddings,
                retrieval_bank,
                settings["retrieval_batch_size"],
            )
            separation = score_separation_metrics(
                evaluation_positive,
                evaluation_locality,
            )
            legacy_gate = calibrate_legacy_positive_gate(
                calibration_positive,
                calibration_locality,
                evaluation_positive,
                evaluation_locality,
                settings["legacy_positive_activation_target"],
            )
            gates = calibrate_global_gates(
                calibration_positive,
                calibration_locality,
                evaluation_positive,
                evaluation_locality,
                settings["locality_far_targets"],
            )
            repeat_results_by_size[bank_size].append({
                "repeat": repeat_index,
                "bank_size": bank_size,
                "query_cohort_fingerprint": cohort_fingerprint,
                "query_owner_count": len(cohort["owner_indices"]),
                "calibration_owner_count": cohort["calibration_owner_count"],
                "calibration_positive_query_count": len(
                    cohort["calibration_positive"]
                ),
                "evaluation_positive_query_count": len(
                    cohort["evaluation_positive"]
                ),
                "calibration_locality_query_count": len(
                    cohort["calibration_locality"]
                ),
                "evaluation_locality_query_count": len(
                    cohort["evaluation_locality"]
                ),
                "retrieval": retrieval_metrics(evaluation_positive),
                "separation": separation,
                "legacy_gate": legacy_gate,
                "gates": gates,
                "diagnostics": routing_diagnostics(
                    cohort["evaluation_positive"],
                    cohort["evaluation_locality"],
                    evaluation_positive,
                    evaluation_locality,
                    gates,
                    bank_indices,
                    records,
                    settings["diagnostic_limit"],
                ),
            })
            del retrieval_bank
            progress.update(1)
        del query_embeddings
    progress.close()

    scales = []
    for requested_size, bank_size in resolved_sizes:
        repeats = repeat_results_by_size[bank_size]
        scales.append({
            "requested_bank_size": requested_size,
            "bank_size": bank_size,
            "repeats": repeats,
            "aggregate": aggregate_global_repeats(repeats),
        })
    result = build_result_header(
        model=model,
        model_key=model_key,
        dataset_metadata=dataset_metadata,
        experiment=experiment,
    )
    result.update({
        "retrieval": "exact_blockwise_cosine",
        "retrieval_dtype": "float32",
        "routing": {
            "legacy_positive_activation_target": settings[
                "legacy_positive_activation_target"
            ],
            "threshold_policy": "per_bank_fixed_calibration_cohort",
            "gate_order": list(GLOBAL_GATE_ORDER),
            "gate_labels": GLOBAL_GATE_LABELS,
            "locality_far_targets": settings["locality_far_targets"],
        },
        "scales": scales,
    })
    write_json_atomic(result, output_path)
    print(f"saved={output_path} scales={len(scales)}")

In [ ]:
def experiment_configuration(
    experiment_mode: str,
    record_limit: int | None,
    record_batch_size: int,
    global_bank_sizes: list[int | None],
    global_repeats: int,
    global_random_seed: int,
    global_legacy_positive_activation_target: float,
    global_query_owner_count: int,
    global_calibration_fraction: float,
    global_positive_calibration_query_limit: int,
    global_positive_evaluation_query_limit: int,
    global_locality_calibration_query_limit: int,
    global_locality_evaluation_query_limit: int,
    global_locality_far_targets: list[float],
    global_diagnostic_limit: int,
    global_retrieval_batch_size: int,
) -> dict[str, Any]:
    validate_experiment_mode(experiment_mode)
    if record_limit is not None and int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    if experiment_mode == "local":
        if record_batch_size <= 0:
            raise ValueError("record_batch_size must be positive.")
        settings = {"record_batch_size": record_batch_size}
    else:
        if global_repeats <= 0:
            raise ValueError("global_repeats must be positive.")
        if not 0.0 < global_legacy_positive_activation_target <= 1.0:
            raise ValueError(
                "global_legacy_positive_activation_target must be in (0, 1]."
            )
        if global_query_owner_count < 2:
            raise ValueError("global_query_owner_count must be at least two.")
        if not 0.0 < global_calibration_fraction < 1.0:
            raise ValueError("global_calibration_fraction must be in (0, 1).")
        query_limits = (
            global_positive_calibration_query_limit,
            global_positive_evaluation_query_limit,
            global_locality_calibration_query_limit,
            global_locality_evaluation_query_limit,
        )
        if any(limit <= 0 for limit in query_limits):
            raise ValueError("Every global query limit must be positive.")
        far_targets = list(dict.fromkeys(float(
            target
        ) for target in global_locality_far_targets))
        if not far_targets or any(
            not 0.0 < target < 1.0 for target in far_targets
        ):
            raise ValueError("global_locality_far_targets must be within (0, 1).")
        if global_diagnostic_limit < 0:
            raise ValueError("global_diagnostic_limit cannot be negative.")
        if global_retrieval_batch_size <= 0:
            raise ValueError("global_retrieval_batch_size must be positive.")
        settings = {
            "bank_sizes": global_bank_sizes,
            "bank_sampling": "fixed_query_owners_with_nested_distractors",
            "query_sampling": "fixed_within_repeat_across_bank_sizes",
            "repeats": global_repeats,
            "random_seed": global_random_seed,
            "legacy_positive_activation_target": (
                global_legacy_positive_activation_target
            ),
            "query_owner_count": global_query_owner_count,
            "calibration_fraction": global_calibration_fraction,
            "positive_calibration_query_limit": (
                global_positive_calibration_query_limit
            ),
            "positive_evaluation_query_limit": (
                global_positive_evaluation_query_limit
            ),
            "locality_calibration_query_limit": (
                global_locality_calibration_query_limit
            ),
            "locality_evaluation_query_limit": (
                global_locality_evaluation_query_limit
            ),
            "locality_far_targets": far_targets,
            "diagnostic_limit": global_diagnostic_limit,
            "retrieval_batch_size": global_retrieval_batch_size,
        }
    return {
        "mode": experiment_mode,
        "record_limit": record_limit,
        "settings": settings,
    }


def build_result_header(
    model,
    model_key: str,
    dataset_metadata: dict[str, Any],
    experiment: dict[str, Any],
) -> dict[str, Any]:
    spec = EMBEDDING_MODEL_SPECS[model_key]
    return {
        "embedding_model_key": model_key,
        "embedding_model": spec["repo"],
        "embedding_backend": spec["backend"],
        "input_prefixes": spec["prefixes"],
        "representation": REPRESENTATION_TYPE,
        "use_fp16": EMBEDDING_USE_FP16,
        "compute_dtype": embedding_compute_dtype(),
        "max_length_cap": EMBEDDING_MAX_LENGTH,
        "effective_max_length": embedding_effective_max_length(model, model_key),
        "dataset": dataset_metadata,
        "experiment": experiment,
    }


def result_matches_configuration(
    output_path: Path,
    model_key: str,
    experiment: dict[str, Any],
    dataset_metadata: dict[str, Any] | None = None,
) -> bool:
    if not output_path.exists():
        return False
    with open(output_path, encoding="utf-8") as file:
        result = json.load(file)
    spec = EMBEDDING_MODEL_SPECS[model_key]
    return (
        result.get("embedding_model_key") == model_key
        and result.get("embedding_model") == spec["repo"]
        and result.get("embedding_backend") == spec["backend"]
        and result.get("input_prefixes") == spec["prefixes"]
        and result.get("representation") == REPRESENTATION_TYPE
        and result.get("use_fp16") == EMBEDDING_USE_FP16
        and result.get("compute_dtype") == embedding_compute_dtype()
        and result.get("max_length_cap") == EMBEDDING_MAX_LENGTH
        and result.get("experiment") == experiment
        and (dataset_metadata is None or result.get("dataset") == dataset_metadata)
    )


def calculate_similarity(
    embedding_model_keys: list[str],
    experiment_mode: str,
    dataset_source: str,
    record_limit: int | None,
    record_batch_size: int,
    overwrite_output: bool,
    global_bank_sizes: list[int | None],
    global_repeats: int,
    global_random_seed: int,
    global_legacy_positive_activation_target: float,
    global_query_owner_count: int,
    global_calibration_fraction: float,
    global_positive_calibration_query_limit: int,
    global_positive_evaluation_query_limit: int,
    global_locality_calibration_query_limit: int,
    global_locality_evaluation_query_limit: int,
    global_locality_far_targets: list[float],
    global_diagnostic_limit: int,
    global_retrieval_batch_size: int,
) -> None:
    model_keys = resolve_embedding_model_keys(embedding_model_keys)
    experiment = experiment_configuration(
        experiment_mode=experiment_mode,
        record_limit=record_limit,
        record_batch_size=record_batch_size,
        global_bank_sizes=global_bank_sizes,
        global_repeats=global_repeats,
        global_random_seed=global_random_seed,
        global_legacy_positive_activation_target=(
            global_legacy_positive_activation_target
        ),
        global_query_owner_count=global_query_owner_count,
        global_calibration_fraction=global_calibration_fraction,
        global_positive_calibration_query_limit=(
            global_positive_calibration_query_limit
        ),
        global_positive_evaluation_query_limit=(
            global_positive_evaluation_query_limit
        ),
        global_locality_calibration_query_limit=(
            global_locality_calibration_query_limit
        ),
        global_locality_evaluation_query_limit=(
            global_locality_evaluation_query_limit
        ),
        global_locality_far_targets=global_locality_far_targets,
        global_diagnostic_limit=global_diagnostic_limit,
        global_retrieval_batch_size=global_retrieval_batch_size,
    )
    dataset = load_similarity_dataset(dataset_source)
    dataset_metadata = similarity_dataset_metadata(dataset, dataset_source)
    calculation_keys = []
    for model_key in model_keys:
        output_path = similarity_output_path(model_key, experiment_mode, record_limit)
        if output_path.exists() and overwrite_output:
            output_path.unlink()
            print(f"removed_for_overwrite={output_path}")
        elif output_path.exists():
            if result_matches_configuration(
                output_path,
                model_key,
                experiment,
                dataset_metadata,
            ):
                print(f"skipped_existing={output_path}")
                continue
            print(f"recalculating_stale={output_path}")
        calculation_keys.append(model_key)
    if not calculation_keys:
        dataset = None
        gc.collect()
        return

    if experiment_mode == "local":
        records = build_local_similarity_records(dataset, record_limit)
    else:
        records = build_global_similarity_records(dataset, record_limit)
    print(
        f"experiment={experiment_mode} source={dataset_source} "
        f"records={len(records)} models={calculation_keys}"
    )
    failures = []
    for model_key in calculation_keys:
        model = None
        try:
            model = load_embedding_model(model_key)
            output_path = similarity_output_path(model_key, experiment_mode, record_limit)
            if experiment_mode == "local":
                run_local_similarity_analysis(
                    dataset,
                    records,
                    model=model,
                    model_key=model_key,
                    output_path=output_path,
                    record_batch_size=record_batch_size,
                    dataset_metadata=dataset_metadata,
                    experiment=experiment,
                )
            else:
                run_global_similarity_analysis(
                    records,
                    model=model,
                    model_key=model_key,
                    output_path=output_path,
                    dataset_metadata=dataset_metadata,
                    experiment=experiment,
                )
        except Exception as error:
            failures.append(f"{model_key}: {type(error).__name__}: {error}")
            print(f"embedding_model_failed={failures[-1]}")
        finally:
            if model is not None:
                move_embedding_model_to_cpu(model, model_key)
            model = None
            clear_embedding_memory(model_key)

    dataset = None
    records = None
    gc.collect()
    if failures:
        raise RuntimeError("Embedding models failed:\n" + "\n".join(failures))

## Tables And Plots

Global visualization starts with a six-stage guide. Retrieval asks whether the correct edit can win when activation is already known to be appropriate. Separation compares positive and locality score distributions without choosing a threshold. The legacy table recreates the old 95%-positive-activation rule and exposes why it may activate on locality. FAR-controlled tables reverse the question by limiting locality mistakes first. The paired table then isolates whether adding top1/top2 margin helps beyond score alone. Finally, bounded failure examples show the actual prompts behind the metrics.

In [ ]:
PAIR_COLORS = {
    "p1-p2": "#0072B2",
    "p1-p3": "#E69F00",
    "p2-p3": "#009E73",
}


def aggregate_pair_table(results: list[tuple[str, dict]], show_std: bool) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        row = {"Embedding model": model_key}
        for pair_key, _, _, label in PAIR_SPECS:
            pair = result["aggregate"]["pairs"][pair_key]
            row[label] = (
                f"{pair['mean']:.4f} +/- {pair['std']:.4f}"
                if show_std
                else f"{pair['mean']:.4f}"
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index("Embedding model")


def observation_count_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        row = {
            "Embedding model": model_key,
            "Records": result["aggregate"]["record_count"],
        }
        for pair_key, _, _, label in PAIR_SPECS:
            row[label] = result["aggregate"]["pairs"][pair_key]["count"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("Embedding model")


def summarize_pairwise_preference(result: dict) -> dict:
    tolerance = 1e-12
    gaps = np.concatenate([
        (
            np.asarray(record["pair_values"]["p1-p2"], dtype=np.float64)[:, None]
            - np.asarray(record["pair_values"]["p1-p3"], dtype=np.float64)[None, :]
        ).reshape(-1)
        for record in result["records"]
    ])
    generality_preferred = int(np.count_nonzero(gaps > tolerance))
    locality_preferred = int(np.count_nonzero(gaps < -tolerance))
    ties = int(gaps.size - generality_preferred - locality_preferred)
    return {
        "comparisons": int(gaps.size),
        "mean_gap": float(gaps.mean()),
        "gap_std": float(gaps.std(ddof=0)),
        "generality_preferred": generality_preferred / gaps.size,
        "locality_preferred": locality_preferred / gaps.size,
        "ties": ties / gaps.size,
    }


def pairwise_preference_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        preference = result["pairwise_preference"]
        rows.append({
            "Embedding model": model_key,
            "Comparisons": preference["comparisons"],
            "Mean gap": preference["mean_gap"],
            "Gap std": preference["gap_std"],
            "Generality preferred": preference["generality_preferred"],
            "Locality preferred": preference["locality_preferred"],
            "Ties": preference["ties"],
        })
    return pd.DataFrame(rows).set_index("Embedding model")


def plot_local_similarity(results: list[tuple[str, dict]], show_std: bool) -> None:
    model_keys = [model_key for model_key, _ in results]
    pair_specs = list(PAIR_SPECS)
    positions = np.arange(len(model_keys), dtype=float)
    group_width = 0.78
    bar_width = group_width / len(pair_specs)
    offsets = (
        np.arange(len(pair_specs), dtype=float) - (len(pair_specs) - 1) / 2
    ) * bar_width

    all_lower = []
    all_upper = []
    figure, axis = plt.subplots(figsize=(18, 7))
    for pair_offset, (pair_key, _, _, label) in zip(offsets, pair_specs):
        means = np.asarray([
            result["aggregate"]["pairs"][pair_key]["mean"]
            for _, result in results
        ])
        stds = np.asarray([
            result["aggregate"]["pairs"][pair_key]["std"]
            for _, result in results
        ])
        axis.bar(
            positions + pair_offset,
            means,
            width=bar_width * 0.92,
            yerr=stds if show_std else None,
            label=f"{pair_key}: {label}",
            color=PAIR_COLORS[pair_key],
            capsize=5 if show_std else 0,
            error_kw={"elinewidth": 1.25, "capthick": 1.25},
        )
        all_lower.extend((means - stds if show_std else means).tolist())
        all_upper.extend((means + stds if show_std else means).tolist())

    y_min = min(0.0, min(all_lower))
    y_max = max(all_upper)
    padding = max(0.05, (y_max - y_min) * 0.12)
    axis.set_ylim(
        max(-1.0, y_min - (padding if y_min < 0 else 0)),
        min(1.0, y_max + padding),
    )
    axis.yaxis.set_major_locator(MaxNLocator(nbins=10))
    axis.set_xticks(positions, model_keys, rotation=20, ha="right")
    axis.set_ylabel("cosine similarity")
    record_counts = {result["aggregate"]["record_count"] for _, result in results}
    count_label = str(record_counts.pop()) if len(record_counts) == 1 else "mixed"
    axis.set_title(f"Local prompt similarity | n={count_label}", fontweight="bold")
    axis.grid(axis="y", alpha=0.22)
    axis.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), title="prompt pair")
    figure.tight_layout(rect=(0, 0, 0.82, 1))
    plt.show()


def mean_std_text(metric: dict[str, float]) -> str:
    return f"{metric['mean']:.4f} +/- {metric['std']:.4f}"



def global_experiment_guide_table() -> pd.DataFrame:
    return pd.DataFrame([
        {
            "Stage": "1. Retrieval",
            "Question": "If this prompt should activate memory, does the correct edit rank first?",
            "Read": "High Recall@1 and MRR are good; no threshold is involved.",
        },
        {
            "Stage": "2. Separation",
            "Question": "Do positive prompts naturally score above no-edit locality prompts?",
            "Read": "AUC near 1 and a positive conservative gap are good.",
        },
        {
            "Stage": "3. Legacy gate",
            "Question": "What happens when the threshold only preserves 95% positive activation?",
            "Read": "This reproduces the old result; locality FAR reveals its cost.",
        },
        {
            "Stage": "4. FAR-controlled score",
            "Question": "How much correct routing survives when locality mistakes are limited first?",
            "Read": "Compare correct routed at 1%, 5%, and 10% target FAR.",
        },
        {
            "Stage": "5. Score + margin",
            "Question": "Does requiring a clear winner over second place improve routing?",
            "Read": "Positive correct-route change and negative FAR/rejection change are good.",
        },
        {
            "Stage": "6. Failures",
            "Question": "Which prompts still retrieve, activate, or reject incorrectly?",
            "Read": "Inspect retrieved edits and scores rather than guessing from averages.",
        },
    ]).set_index("Stage")


def global_separation_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    metric_labels = (
        ("score_auc", "Score AUC"),
        ("margin_auc", "Margin AUC"),
        ("positive_score_p05", "Positive score p05"),
        ("locality_score_p95", "Locality score p95"),
        ("score_conservative_gap", "Score p05 - locality p95"),
        ("positive_margin_p05", "Positive margin p05"),
        ("locality_margin_p95", "Locality margin p95"),
        ("margin_conservative_gap", "Margin p05 - locality p95"),
    )
    for model_key, result in results:
        for scale in result["scales"]:
            separation = scale["aggregate"]["separation"]
            row = {
                "Embedding model": model_key,
                "Bank size": scale["bank_size"],
            }
            for metric_key, label in metric_labels:
                row[label] = mean_std_text(separation[metric_key])
            rows.append(row)
    return pd.DataFrame(rows).set_index(["Embedding model", "Bank size"])


def global_legacy_gate_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        target = result["routing"]["legacy_positive_activation_target"]
        for scale in result["scales"]:
            legacy = scale["aggregate"]["legacy_gate"]
            rows.append({
                "Embedding model": model_key,
                "Bank size": scale["bank_size"],
                "Target positive activation": target,
                "Positive activation": mean_std_text(
                    legacy["positive_activation_rate"]
                ),
                "Correct routed": mean_std_text(legacy["correct_routed_rate"]),
                "Incorrect routed": mean_std_text(
                    legacy["incorrect_routed_rate"]
                ),
                "Locality FAR": mean_std_text(
                    legacy["locality_false_activation_rate"]
                ),
                "Score threshold": mean_std_text(legacy["score_threshold"]),
            })
    return pd.DataFrame(rows).set_index(["Embedding model", "Bank size"])


def global_retrieval_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    metric_labels = (
        ("recall_at_1", "Recall@1"),
        ("recall_at_5", "Recall@5"),
        ("recall_at_10", "Recall@10"),
        ("mrr", "MRR"),
        ("correct_margin_mean", "Correct margin"),
        ("top1_margin_mean", "Top1/top2 margin"),
    )
    for model_key, result in results:
        for scale in result["scales"]:
            retrieval = scale["aggregate"]["retrieval"]
            row = {
                "Embedding model": model_key,
                "Bank size": scale["bank_size"],
                "Repeats": len(scale["repeats"]),
            }
            for metric_key, label in metric_labels:
                row[label] = mean_std_text(retrieval[metric_key])
            rows.append(row)
    return pd.DataFrame(rows).set_index(["Embedding model", "Bank size"])


def global_gate_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        gate_labels = result["routing"]["gate_labels"]
        for scale in result["scales"]:
            for gate_key in result["routing"]["gate_order"]:
                for target_key, metrics in scale["aggregate"]["gates"][gate_key].items():
                    rows.append({
                        "Embedding model": model_key,
                        "Bank size": scale["bank_size"],
                        "Gate": gate_labels[gate_key],
                        "Target locality FAR": float(target_key),
                        "Correct routed": mean_std_text(
                            metrics["correct_routed_rate"]
                        ),
                        "Positive rejected": mean_std_text(
                            metrics["positive_rejection_rate"]
                        ),
                        "Incorrect routed": mean_std_text(
                            metrics["incorrect_routed_rate"]
                        ),
                        "Actual locality FAR": mean_std_text(
                            metrics["locality_false_activation_rate"]
                        ),
                        "Score threshold": mean_std_text(
                            metrics["score_threshold"]
                        ),
                        "Margin threshold": mean_std_text(
                            metrics["margin_threshold"]
                        ),
                    })
    return pd.DataFrame(rows).set_index([
        "Embedding model",
        "Bank size",
        "Gate",
        "Target locality FAR",
    ])


def global_query_count_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        for scale in result["scales"]:
            repeats = scale["repeats"]
            rows.append({
                "Embedding model": model_key,
                "Bank size": scale["bank_size"],
                "Query owners": int(np.mean([
                    repeat["query_owner_count"] for repeat in repeats
                ])),
                "Calibration positives": int(np.mean([
                    repeat["calibration_positive_query_count"]
                    for repeat in repeats
                ])),
                "Evaluation positives": int(np.mean([
                    repeat["evaluation_positive_query_count"]
                    for repeat in repeats
                ])),
                "Calibration locality": int(np.mean([
                    repeat["calibration_locality_query_count"]
                    for repeat in repeats
                ])),
                "Evaluation locality": int(np.mean([
                    repeat["evaluation_locality_query_count"]
                    for repeat in repeats
                ])),
            })
    return pd.DataFrame(rows).set_index(["Embedding model", "Bank size"])


def global_gate_comparison_table(
    results: list[tuple[str, dict]],
) -> pd.DataFrame:
    rows = []
    comparison_metrics = (
        ("correct_routed_rate", "Correct routed change"),
        ("positive_rejection_rate", "Positive rejection change"),
        ("locality_false_activation_rate", "Locality FAR change"),
    )
    for model_key, result in results:
        for scale in result["scales"]:
            for target_key in scale["repeats"][0]["gates"]["score"]:
                row = {
                    "Embedding model": model_key,
                    "Bank size": scale["bank_size"],
                    "Target locality FAR": float(target_key),
                }
                for metric_key, label in comparison_metrics:
                    differences = np.asarray([
                        repeat["gates"]["score_margin"][target_key][metric_key]
                        - repeat["gates"]["score"][target_key][metric_key]
                        for repeat in scale["repeats"]
                    ])
                    row[label] = (
                        f"{differences.mean():+.4f} +/- "
                        f"{differences.std(ddof=0):.4f}"
                    )
                rows.append(row)
    return pd.DataFrame(rows).set_index([
        "Embedding model",
        "Bank size",
        "Target locality FAR",
    ])



def plot_global_separation(results: list[tuple[str, dict]]) -> None:
    plot_specs = (
        ("score_auc", "Score AUC", True),
        ("margin_auc", "Margin AUC", True),
        ("score_conservative_gap", "Score conservative gap", False),
        ("margin_conservative_gap", "Margin conservative gap", False),
    )
    figure, axes = plt.subplots(2, 2, figsize=(18, 11), sharex=True)
    for axis, (metric_key, label, bounded) in zip(axes.flat, plot_specs):
        for model_key, result in results:
            sizes = np.asarray([scale["bank_size"] for scale in result["scales"]])
            means = np.asarray([
                scale["aggregate"]["separation"][metric_key]["mean"]
                for scale in result["scales"]
            ])
            stds = np.asarray([
                scale["aggregate"]["separation"][metric_key]["std"]
                for scale in result["scales"]
            ])
            axis.errorbar(
                sizes,
                means,
                yerr=stds,
                marker="o",
                linewidth=2,
                capsize=4,
                label=model_key,
            )
        axis.set_xscale("log")
        axis.set_ylabel(label)
        axis.grid(alpha=0.22)
        if bounded:
            axis.set_ylim(0.0, 1.02)
        else:
            axis.axhline(0.0, color="#D55E00", linestyle="--", linewidth=1.5)
    for axis in axes[-1]:
        axis.set_xlabel("edit-bank size (log scale)")
    axes[0, 1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    figure.suptitle(
        "Positive versus locality separation | no threshold",
        fontweight="bold",
    )
    figure.tight_layout(rect=(0, 0, 0.88, 0.96))
    plt.show()


def plot_global_legacy_gate(results: list[tuple[str, dict]]) -> None:
    figure, axes = plt.subplots(1, 2, figsize=(18, 7), sharex=True)
    for axis, (metric_key, label) in zip(
        axes,
        (
            ("correct_routed_rate", "Correct routed rate"),
            ("locality_false_activation_rate", "Locality false activation"),
        ),
    ):
        for model_key, result in results:
            sizes = np.asarray([scale["bank_size"] for scale in result["scales"]])
            means = np.asarray([
                scale["aggregate"]["legacy_gate"][metric_key]["mean"]
                for scale in result["scales"]
            ])
            stds = np.asarray([
                scale["aggregate"]["legacy_gate"][metric_key]["std"]
                for scale in result["scales"]
            ])
            axis.errorbar(
                sizes,
                means,
                yerr=stds,
                marker="o",
                linewidth=2,
                capsize=4,
                label=model_key,
            )
        axis.set_xscale("log")
        axis.set_xlabel("edit-bank size (log scale)")
        axis.set_ylabel(label)
        axis.set_ylim(0.0, 1.02)
        axis.grid(alpha=0.22)
    axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    figure.suptitle(
        "Legacy gate | threshold preserves 95% positive activation",
        fontweight="bold",
    )
    figure.tight_layout(rect=(0, 0, 0.88, 0.95))
    plt.show()


def plot_global_retrieval(results: list[tuple[str, dict]]) -> None:
    figure, axes = plt.subplots(1, 2, figsize=(18, 7), sharex=True)
    for axis, (metric_key, label) in zip(
        axes,
        (("recall_at_1", "Recall@1"), ("mrr", "MRR")),
    ):
        for model_key, result in results:
            sizes = np.asarray([scale["bank_size"] for scale in result["scales"]])
            means = np.asarray([
                scale["aggregate"]["retrieval"][metric_key]["mean"]
                for scale in result["scales"]
            ])
            stds = np.asarray([
                scale["aggregate"]["retrieval"][metric_key]["std"]
                for scale in result["scales"]
            ])
            axis.errorbar(
                sizes,
                means,
                yerr=stds,
                marker="o",
                linewidth=2,
                capsize=4,
                label=model_key,
            )
        axis.set_xscale("log")
        axis.set_xlabel("edit-bank size (log scale)")
        axis.set_ylabel(label)
        axis.set_ylim(0.0, 1.02)
        axis.grid(alpha=0.22)
    axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    figure.suptitle(
        "Global retrieval | fixed queries, growing distractor bank",
        fontweight="bold",
    )
    figure.tight_layout(rect=(0, 0, 0.88, 0.95))
    plt.show()


def plot_global_gate_results(results: list[tuple[str, dict]]) -> None:
    gate_order = results[0][1]["routing"]["gate_order"]
    target_keys = [
        far_target_key(target)
        for target in results[0][1]["routing"]["locality_far_targets"]
    ]
    for gate_key in gate_order:
        figure, axes = plt.subplots(
            2,
            len(target_keys),
            figsize=(6 * len(target_keys), 10),
            sharex=True,
        )
        axes = np.asarray(axes).reshape(2, len(target_keys))
        for column, target_key in enumerate(target_keys):
            target = float(target_key)
            for model_key, result in results:
                sizes = np.asarray([
                    scale["bank_size"] for scale in result["scales"]
                ])
                correct_means = np.asarray([
                    scale["aggregate"]["gates"][gate_key][target_key][
                        "correct_routed_rate"
                    ]["mean"]
                    for scale in result["scales"]
                ])
                correct_stds = np.asarray([
                    scale["aggregate"]["gates"][gate_key][target_key][
                        "correct_routed_rate"
                    ]["std"]
                    for scale in result["scales"]
                ])
                locality_means = np.asarray([
                    scale["aggregate"]["gates"][gate_key][target_key][
                        "locality_false_activation_rate"
                    ]["mean"]
                    for scale in result["scales"]
                ])
                locality_stds = np.asarray([
                    scale["aggregate"]["gates"][gate_key][target_key][
                        "locality_false_activation_rate"
                    ]["std"]
                    for scale in result["scales"]
                ])
                axes[0, column].errorbar(
                    sizes,
                    correct_means,
                    yerr=correct_stds,
                    marker="o",
                    linewidth=2,
                    capsize=4,
                    label=model_key,
                )
                axes[1, column].errorbar(
                    sizes,
                    locality_means,
                    yerr=locality_stds,
                    marker="o",
                    linewidth=2,
                    capsize=4,
                    label=model_key,
                )
            axes[0, column].set_title(f"target locality FAR = {target:.0%}")
            axes[0, column].set_ylabel("correct routed rate")
            axes[1, column].set_ylabel("actual locality FAR")
            axes[1, column].axhline(
                target,
                color="#D55E00",
                linestyle="--",
                linewidth=1.5,
                label="target",
            )
            for row in range(2):
                axes[row, column].set_xscale("log")
                axes[row, column].set_ylim(0.0, 1.02)
                axes[row, column].grid(alpha=0.22)
            axes[1, column].set_xlabel("edit-bank size (log scale)")
        axes[0, -1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
        figure.suptitle(
            f"Routing gate: {GLOBAL_GATE_LABELS[gate_key]}",
            fontweight="bold",
        )
        figure.tight_layout(rect=(0, 0, 0.9, 0.96))
        plt.show()



def global_diagnostic_examples_table(
    results: list[tuple[str, dict]],
    bank_size: int,
    gate_key: str,
    far_target: float,
) -> pd.DataFrame:
    target_key = far_target_key(far_target)
    rows = []
    for model_key, result in results:
        scale = next(
            (scale for scale in result["scales"] if scale["bank_size"] == bank_size),
            result["scales"][-1],
        )
        repeat = scale["repeats"][0]
        categories = {
            "wrong top1": repeat["diagnostics"]["wrong_top1"],
            "rejected positive": repeat["diagnostics"]["gates"][gate_key][
                target_key
            ]["rejected_positive"],
            "false locality": repeat["diagnostics"]["gates"][gate_key][
                target_key
            ]["false_locality"],
        }
        for category, examples in categories.items():
            for example in examples:
                rows.append({
                    "Embedding model": model_key,
                    "Bank size": scale["bank_size"],
                    "Failure": category,
                    "Query": example["query"],
                    "Expected edit": example.get("expected_edit", "--"),
                    "Retrieved edit": example["retrieved_edit"],
                    "Top1 score": example["top1_score"],
                    "Top1/top2 margin": example["top1_margin"],
                })
    if not rows:
        return pd.DataFrame(columns=[
            "Embedding model",
            "Bank size",
            "Failure",
            "Query",
            "Expected edit",
            "Retrieved edit",
            "Top1 score",
            "Top1/top2 margin",
        ])
    return pd.DataFrame(rows).set_index([
        "Embedding model",
        "Bank size",
        "Failure",
    ])


def visualize_similarity(
    embedding_model_keys: list[str],
    experiment_mode: str,
    dataset_source: str,
    record_limit: int | None,
    show_std: bool,
    experiment: dict[str, Any],
    diagnostic_bank_size: int,
    diagnostic_gate: str,
    diagnostic_far_target: float,
) -> None:
    model_keys = resolve_embedding_model_keys(embedding_model_keys)
    dataset = load_similarity_dataset(dataset_source)
    dataset_metadata = similarity_dataset_metadata(dataset, dataset_source)
    results = []
    for model_key in model_keys:
        output_path = similarity_output_path(model_key, experiment_mode, record_limit)
        if not output_path.exists():
            raise FileNotFoundError(
                f"Embedding result does not exist: {output_path}. Calculate it first."
            )
        if not result_matches_configuration(
            output_path,
            model_key,
            experiment,
            dataset_metadata,
        ):
            raise ValueError(f"Stale embedding result: {output_path}. Recalculate it.")
        with open(output_path, encoding="utf-8") as file:
            result = json.load(file)
        if experiment_mode == "local":
            result["pairwise_preference"] = summarize_pairwise_preference(result)
            result.pop("records", None)
        results.append((model_key, result))
    dataset = None
    gc.collect()

    if experiment_mode == "local":
        display(aggregate_pair_table(results, show_std=show_std).style)
        display(
            pairwise_preference_table(results).style.format({
                "Comparisons": "{:,}",
                "Mean gap": "{:.4f}",
                "Gap std": "{:.4f}",
                "Generality preferred": "{:.4f}",
                "Locality preferred": "{:.4f}",
                "Ties": "{:.4f}",
            })
        )
        display(observation_count_table(results).style.format("{:,}"))
        plot_local_similarity(results, show_std=show_std)
    else:
        display(global_experiment_guide_table().style)
        display(global_retrieval_table(results).style)
        display(global_separation_table(results).style)
        display(global_legacy_gate_table(results).style)
        display(global_gate_table(results).style)
        display(global_gate_comparison_table(results).style)
        display(global_query_count_table(results).style.format("{:,}"))
        display(global_diagnostic_examples_table(
            results,
            bank_size=diagnostic_bank_size,
            gate_key=diagnostic_gate,
            far_target=diagnostic_far_target,
        ).style)
        plot_global_retrieval(results)
        plot_global_separation(results)
        plot_global_legacy_gate(results)
        plot_global_gate_results(results)

## Experiment Configuration

The final global run keeps E5-base, E5-large, and BGE-M3: an efficient retrieval model, its larger quality-oriented counterpart, and the strongest prior retrieval/abstention balance. Banks start at 1,000 because smaller banks cannot support a stable fixed 1,000-owner calibration/evaluation cohort. FP16 is retained because the completed precision audit found no meaningful ranking change.

In [ ]:
# Choose one experiment without duplicating the calculation/visualization pipeline.
EXPERIMENT_MODE = "global"  # "local" or "global"
DATASET_SOURCE = "huggingface"  # Use "local" for the matching local Arrow artifact.

# Comment out a line to disable it; list order controls calculation and display order.
EMBEDDING_MODEL_KEYS = [
    # "all-mpnet-base-v2",
    # "all-MiniLM-L6-v2",
    # "all-MiniLM-L12-v2",
    "e5-base-v2",
    "e5-large-v2",
    # "bge-base-en-v1.5",
    # "bge-large-en-v1.5",
    "bge-m3",
]

EMBEDDING_GPU_ID = 0
EMBEDDING_USE_FP16 = True
EMBEDDING_MAX_LENGTH = 384
EMBEDDING_BATCH_SIZE = 64
RECORD_BATCH_SIZE = 64  # Local mode only.
RECORD_LIMIT = None  # Set a positive integer for diagnostic result files.
OVERWRITE_EXISTING_RESULT = False

# Global banks share the same 1,000 query owners within each repeat.
GLOBAL_BANK_SIZES = [
    1000,
    2000,
    5000,
    10000,
    None,  # Complete 63,441-entry bank.
]
GLOBAL_REPEATS = 3
GLOBAL_RANDOM_SEED = 42
GLOBAL_LEGACY_POSITIVE_ACTIVATION_TARGET = 0.95
GLOBAL_QUERY_OWNER_COUNT = 1000
GLOBAL_CALIBRATION_FRACTION = 0.2
GLOBAL_POSITIVE_CALIBRATION_QUERY_LIMIT = 400
GLOBAL_POSITIVE_EVALUATION_QUERY_LIMIT = 1600
GLOBAL_LOCALITY_CALIBRATION_QUERY_LIMIT = 1000
GLOBAL_LOCALITY_EVALUATION_QUERY_LIMIT = 2000
GLOBAL_LOCALITY_FAR_TARGETS = [
    0.01,
    0.05,
    0.10,
]
GLOBAL_DIAGNOSTIC_LIMIT = 5
GLOBAL_RETRIEVAL_BATCH_SIZE = 256

PLOT_SHOW_STD = True
GLOBAL_DIAGNOSTIC_DISPLAY_BANK_SIZE = 5000
GLOBAL_DIAGNOSTIC_DISPLAY_GATE = "score_margin"
GLOBAL_DIAGNOSTIC_DISPLAY_FAR_TARGET = 0.05

EXPERIMENT = experiment_configuration(
    experiment_mode=EXPERIMENT_MODE,
    record_limit=RECORD_LIMIT,
    record_batch_size=RECORD_BATCH_SIZE,
    global_bank_sizes=GLOBAL_BANK_SIZES,
    global_repeats=GLOBAL_REPEATS,
    global_random_seed=GLOBAL_RANDOM_SEED,
    global_legacy_positive_activation_target=(
        GLOBAL_LEGACY_POSITIVE_ACTIVATION_TARGET
    ),
    global_query_owner_count=GLOBAL_QUERY_OWNER_COUNT,
    global_calibration_fraction=GLOBAL_CALIBRATION_FRACTION,
    global_positive_calibration_query_limit=(
        GLOBAL_POSITIVE_CALIBRATION_QUERY_LIMIT
    ),
    global_positive_evaluation_query_limit=(
        GLOBAL_POSITIVE_EVALUATION_QUERY_LIMIT
    ),
    global_locality_calibration_query_limit=(
        GLOBAL_LOCALITY_CALIBRATION_QUERY_LIMIT
    ),
    global_locality_evaluation_query_limit=(
        GLOBAL_LOCALITY_EVALUATION_QUERY_LIMIT
    ),
    global_locality_far_targets=GLOBAL_LOCALITY_FAR_TARGETS,
    global_diagnostic_limit=GLOBAL_DIAGNOSTIC_LIMIT,
    global_retrieval_batch_size=GLOBAL_RETRIEVAL_BATCH_SIZE,
)

## Manual Actions

Calculation and visualization remain explicit. Existing global files from the earlier positive-only threshold experiment are detected as stale because the experiment settings and result schema have changed.

In [ ]:
# Calculate and save one JSON per enabled embedding model:
# calculate_similarity(
#     embedding_model_keys=EMBEDDING_MODEL_KEYS,
#     experiment_mode=EXPERIMENT_MODE,
#     dataset_source=DATASET_SOURCE,
#     record_limit=RECORD_LIMIT,
#     record_batch_size=RECORD_BATCH_SIZE,
#     overwrite_output=OVERWRITE_EXISTING_RESULT,
#     global_bank_sizes=GLOBAL_BANK_SIZES,
#     global_repeats=GLOBAL_REPEATS,
#     global_random_seed=GLOBAL_RANDOM_SEED,
#     global_legacy_positive_activation_target=(
#         GLOBAL_LEGACY_POSITIVE_ACTIVATION_TARGET
#     ),
#     global_query_owner_count=GLOBAL_QUERY_OWNER_COUNT,
#     global_calibration_fraction=GLOBAL_CALIBRATION_FRACTION,
#     global_positive_calibration_query_limit=(
#         GLOBAL_POSITIVE_CALIBRATION_QUERY_LIMIT
#     ),
#     global_positive_evaluation_query_limit=(
#         GLOBAL_POSITIVE_EVALUATION_QUERY_LIMIT
#     ),
#     global_locality_calibration_query_limit=(
#         GLOBAL_LOCALITY_CALIBRATION_QUERY_LIMIT
#     ),
#     global_locality_evaluation_query_limit=(
#         GLOBAL_LOCALITY_EVALUATION_QUERY_LIMIT
#     ),
#     global_locality_far_targets=GLOBAL_LOCALITY_FAR_TARGETS,
#     global_diagnostic_limit=GLOBAL_DIAGNOSTIC_LIMIT,
#     global_retrieval_batch_size=GLOBAL_RETRIEVAL_BATCH_SIZE,
# )

# Visualize enabled results in configured model and bank-size order:
# visualize_similarity(
#     embedding_model_keys=EMBEDDING_MODEL_KEYS,
#     experiment_mode=EXPERIMENT_MODE,
#     dataset_source=DATASET_SOURCE,
#     record_limit=RECORD_LIMIT,
#     show_std=PLOT_SHOW_STD,
#     experiment=EXPERIMENT,
#     diagnostic_bank_size=GLOBAL_DIAGNOSTIC_DISPLAY_BANK_SIZE,
#     diagnostic_gate=GLOBAL_DIAGNOSTIC_DISPLAY_GATE,
#     diagnostic_far_target=GLOBAL_DIAGNOSTIC_DISPLAY_FAR_TARGET,
# )